# Week 8-1 · Tutorial — Doubt-Solving on Python after DMP-01
**Module:** Data Analysis & Modeling in Python (tutorial) · Instructor: Manusha Rao.

A hands-on clinic reinforcing DMP-01: **fetching data** (yfinance gotchas + CSV round-trip), **time zones**,
**vectorization vs iteration**, and a full **moving-average crossover back-test** (SMA 21 / LMA 63).

> The live parts use `yfinance` (not installed here), so we work from the **shipped NIFTY-50 daily file**
> `nifty50_daily.csv` (962 bars, 2017 → 2020). The yfinance-specific points are explained and shown as the
> exact code you'd run.

## 1. The core libraries (one line each)
| Library | Used for |
|---|---|
| `pandas` | data analysis & manipulation (DataFrames) |
| `numpy` | arrays, linear algebra, vectorized maths |
| `datetime` | building/​manipulating dates & times |
| `yfinance` | downloading OHLCV data for tickers |
| `warnings` | silence non-fatal warning messages |
| `matplotlib` | plotting & visualization |
| `cufflinks` | bridge pandas → Plotly for interactive `.iplot()` charts |

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import datetime as dt
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
print("pandas", pd.__version__)

## 2. A date is not a string, and not a tuple
`start = (2017, 1, 1)` is a **tuple** (round brackets); `"2017-01-01"` is a **string**. `yfinance.download`
needs real **date** objects, so passing a tuple raises *`'tuple' object has no attribute 'timestamp'`*.
Build dates with `datetime`.

In [ ]:
maybe_tuple = (2017, 1, 1)
print("type of (2017,1,1):", type(maybe_tuple).__name__)        # tuple
print("type of '2017-01-01':", type("2017-01-01").__name__)      # str
start_date = dt.date(2017, 1, 1)
end_date   = dt.date(2020, 11, 27)
print("type of dt.date(...):", type(start_date).__name__, "->", start_date)

### The yfinance call (for reference)
Two arguments matter a lot:
- `multi_level_index=False` — flattens the new multi-level columns (no `('Close','^NSEI')` mess).
- `auto_adjust` — **True** (default) gives corporate-action-adjusted prices; **False** adds a separate
  `Adj Close` column and leaves OHLC unadjusted.
```python
import yfinance as yf
df = yf.download("^NSEI", start=start_date, end=end_date,
                 multi_level_index=False, auto_adjust=True)
# intraday (note: Yahoo only serves the last 60 days for intraday):
intraday = yf.download("^NSEI", period="60d", interval="5m",
                       multi_level_index=False)
```

## 3. Load the shipped CSV the right way
We read the same NIFTY data from disk. First the *wrong* way to expose the defaults, then the clean way.

In [ ]:
# plain read: Date is just a column, index is a RangeIndex
raw = pd.read_csv("nifty50_daily.csv")
print("plain read -> index:", type(raw.index).__name__, "| columns:", list(raw.columns))
raw.info()

### `index_col` + `parse_dates`
`index_col=0` (or `'Date'`) makes Date the index; `parse_dates=True` turns the date strings into a real
`DatetimeIndex`. Either the column **number** or its **name** works.

In [ ]:
df = pd.read_csv("nifty50_daily.csv", index_col="Date", parse_dates=True)
print("clean read -> index:", type(df.index).__name__)
print("shape:", df.shape, "| df.shape[0] rows =", df.shape[0], ", df.shape[1] cols =", df.shape[1])
print(df.head(3))

### Saving back out — `to_csv`
Round-tripping to disk is the lecture's contingency tip: download once, save, reload offline.
```python
df.to_csv("my_data.csv")     # any name you like
```

In [ ]:
df.to_csv("_roundtrip_check.csv")
back = pd.read_csv("_roundtrip_check.csv", index_col="Date", parse_dates=True)
import os; os.remove("_roundtrip_check.csv")
print("round-trip identical shape:", back.shape == df.shape)

### Windows file paths
A backslash starts an escape sequence, so a raw Windows path breaks. Use a **raw string** `r"..."` or
**double backslashes**:
```python
pd.read_csv(r"C:\Users\me\Downloads\data.csv")   # raw string
pd.read_csv("C:\\Users\\me\\Downloads\\data.csv") # doubled
```

## 4. Adjusted vs unadjusted close
`Adj Close` bakes in **corporate actions** — dividends, stock splits, issuances, mergers — applied
*backwards* over history so returns are continuous. For an **index** like NIFTY-50 there are no splits, so
Adj Close ≈ Close. For long-horizon analysis, prefer **adjusted** close.

## 5. Time zones
yfinance intraday timestamps arrive in **UTC** (`+00:00`). Convert with `tz_convert`; list zones via `pytz`
(or the stdlib `zoneinfo`, used here).
```python
intraday.index = intraday.index.tz_convert("Asia/Kolkata")  # IST
import pytz; pytz.all_timezones[:3]
```

In [ ]:
from zoneinfo import available_timezones
zones = available_timezones()
print("total tz names:", len(zones))
print("a few:", sorted(z for z in zones if "Kolkata" in z or "London" in z)[:3])
# demonstrate a UTC -> IST conversion on a sample timestamp index
idx = pd.date_range("2020-11-27 09:30", periods=3, freq="5min", tz="UTC")
print("UTC :", [t.strftime("%H:%M") for t in idx])
print("IST :", [t.strftime("%H:%M") for t in idx.tz_convert("Asia/Kolkata")])

## 6. Vectorization vs iteration (the event-driven analogy)
A **list** `* 3` *repeats* the list; a **numpy array** `* 3` multiplies every element **at once**
(vectorized). Doing it element-by-element needs a `for` loop — the same shape as **event-driven** back-testing.

In [ ]:
number_list = [1, 2, 11, 5, 8]
print("list * 3  (repeat):", number_list * 3)
arr = np.array(number_list)
print("array * 3 (vectorized):", arr * 3)
loop = [v * 3 for v in number_list]        # event-driven style: one element at a time
print("for-loop  (iterated):", loop)

## 7. The moving-average crossover back-test
**Rule:** go **long** when the short MA crosses **above** the long MA; go **short/flat** when it crosses below.
Windows: short `SMA = 21`, long `LMA = 63`. We run all 7 vectorized steps: data → indicators → signal →
position → returns → equity → analysis.

In [ ]:
data = pd.read_csv("nifty50_daily.csv", index_col="Date", parse_dates=True)[["Close"]].copy()
SW, LW = 21, 63
data["SMA"] = data["Close"].rolling(SW).mean()
data["LMA"] = data["Close"].rolling(LW).mean()
data = data.dropna()
print("rows after warm-up:", data.shape[0])
print(data.head(2))

### Signal on the crossover (today vs yesterday)
Long entry: today `SMA > LMA` **and** yesterday `SMA <= LMA`. We capture the *state* (long when SMA>LMA) and
detect crossings with `.diff()`.

In [ ]:
data["regime"] = np.where(data["SMA"] > data["LMA"], 1, -1)   # +1 long, -1 short
data["signal"] = data["regime"].diff()                        # +2 = golden cross, -2 = death cross
crosses = data["signal"].value_counts().to_dict()
print("crossings (2.0=golden up, -2.0=death down):", crosses)

### Position, shift, returns vs buy-and-hold

In [ ]:
data["position"]  = data["regime"].shift(1)          # act next bar (no look-ahead)
data["cc"]        = data["Close"].pct_change()
data = data.dropna()
data["strat"]     = data["position"] * data["cc"]
data["bh_eq"]     = (1 + data["cc"]).cumprod()
data["str_eq"]    = (1 + data["strat"]).cumprod()
bh  = data["bh_eq"].iloc[-1] - 1
st  = data["str_eq"].iloc[-1] - 1
print(f"period {data.index[0].date()} -> {data.index[-1].date()}")
print(f"Buy & Hold     : {bh*100:.2f}%")
print(f"SMA21/LMA63 x  : {st*100:.2f}%")
print("time long:", int((data['position']==1).sum()), "| time short:", int((data['position']==-1).sum()))

### Equity curves

In [ ]:
fig, ax = plt.subplots(figsize=(8,3.6))
ax.plot(data.index, data["bh_eq"],  label=f"Buy & Hold ({bh*100:.1f}%)", color="#7f1d1d", lw=1.3)
ax.plot(data.index, data["str_eq"], label=f"Crossover ({st*100:.1f}%)", color="#dc2626", lw=1.3)
ax.axhline(1, color="gray", ls="--", lw=0.7)
ax.set_title("NIFTY-50 — Buy & Hold vs SMA21/LMA63 crossover")
ax.set_ylabel("Growth of 1"); ax.legend(); ax.grid(alpha=0.25)
fig.tight_layout(); fig.savefig("xover_preview.png", dpi=110)
print("equity curve drawn")

## Key takeaways
- **Build dates with `datetime`** — a tuple `(2017,1,1)` or a string isn't a date; `yf.download` needs real date objects.
- **yfinance flags:** `multi_level_index=False` (flatten columns), `auto_adjust=True` (corporate-action-adjusted prices; `Adj Close` only appears when `False`). Intraday is limited to the **last 60 days**.
- **CSV round-trip:** `to_csv` to save, `read_csv(index_col=..., parse_dates=True)` to reload with a proper `DatetimeIndex`. Use raw strings `r"..."` for Windows paths.
- **Time zones:** yfinance intraday is UTC; `tz_convert("Asia/Kolkata")` for IST; `pytz.all_timezones` lists them.
- **Vectorization:** array ops run all at once (fast); a `for` loop is element-by-element — the shape of event-driven back-testing.
- **MA crossover** in 7 vectorized steps: indicators (`rolling().mean()`) → regime (`np.where`) → crossing (`.diff()`) → position (`.shift(1)`) → returns (`position * pct_change`) → equity (`cumprod`) → compare to buy-and-hold.

# Additive validation layer

The cells below were appended to this validated copy only. The original notebook file is unchanged.

In [ ]:
import pandas as pd
from pathlib import Path
base = Path('.')
files = ['dmp01_tutorial_html_audit.csv', 'dmp01_tutorial_data_inventory.csv', 'dmp01_tutorial_notebook_metrics.csv', 'dmp01_tutorial_csv_roundtrip_timezone.csv', 'dmp01_tutorial_vectorization_checks.csv', 'dmp01_tutorial_crossover_metrics.csv', 'dmp01_tutorial_validation_controls.csv']
data = {f: pd.read_csv(base / f) for f in files}
{k: v.shape for k, v in data.items()}

## 1. Source and notebook audit

In [ ]:
print(data['dmp01_tutorial_data_inventory.csv'].to_string(index=False))
print(data['dmp01_tutorial_notebook_metrics.csv'].to_string(index=False))
metrics = data['dmp01_tutorial_notebook_metrics.csv'].set_index('metric')
assert int(metrics.loc['original_execution_errors','value']) == 0

## 2. CSV, timezone, and vectorization controls

In [ ]:
print(data['dmp01_tutorial_csv_roundtrip_timezone.csv'].to_string(index=False))
print(data['dmp01_tutorial_vectorization_checks.csv'].to_string(index=False))
checks = data['dmp01_tutorial_csv_roundtrip_timezone.csv'].set_index('check')
assert checks.loc['clean_read_index','value'] == 'DatetimeIndex'
assert checks.loc['roundtrip_same_shape','value'] in [True, 'True']

## 3. Crossover metrics

In [ ]:
print(data['dmp01_tutorial_crossover_metrics.csv'].to_string(index=False))
cross = data['dmp01_tutorial_crossover_metrics.csv'].set_index('metric')
assert int(cross.loc['source_rows','value']) == 962
assert int(cross.loc['rows_after_21_63_ma_warmup','value']) == 900
assert int(cross.loc['rows_after_return_and_position_dropna','value']) == 899

## 4. Validation controls

In [ ]:
print(data['dmp01_tutorial_validation_controls.csv'].to_string(index=False))
assert data['dmp01_tutorial_validation_controls.csv'].shape[0] >= 6